# tiny-voca -> Roman Urdu (1000 UrduTTS clips)
Kaggle: turn **Internet ON** and pick a **GPU** accelerator. Run top to bottom.

In [ ]:
STEPS = 6000      # fine-tune steps
N_TRAIN = 1000
N_VAL = 32

In [ ]:
%%bash
mkdir -p /kaggle/working/tiny-urdu && cd /kaggle/working/tiny-urdu
[ -d tiny-voca ] || git clone -q https://github.com/n4ze3m/tiny-voca
git -C tiny-voca checkout -q ca93ad1e35a834a9247347c6821cfeb4c82c6a96
pip install -q --no-deps -e tiny-voca
pip install -q librosa pyarrow "huggingface-hub>=0.34" "transformers>=4.40,<5" scipy datasets soundfile
hf download facebook/wav2vec2-lv-60-espeak-cv-ft > /dev/null
hf download n4ze3m/tiny-voca tiny-voca.pt --local-dir weights > /dev/null
echo ok

In [ ]:
%cd /kaggle/working/tiny-urdu
!mkdir -p urdu runs

## Urdu front end + helper scripts

In [ ]:
%%writefile urdu/frontend.py
"""Urdu front end for tiny-voca.

tiny-voca has a fixed inventory of 76 English phoneme symbols. Instead of
growing it, this module squeezes Urdu into that inventory, so the English
checkpoint can be fine-tuned (embeddings and all) rather than trained from zero.

Two kinds of input are accepted by ``phonemize``:

* ``"ipa:<phonemes>"``  The UrduTTS ``phonemes`` column. Used for TRAINING:
  it is what was actually said in the audio. Stress marks are dropped
  (see below).
* anything else         Roman Urdu such as ``"aap kaise hain?"``. Used for
  INFERENCE, through the rule-based ``roman_to_phones`` G2P.

Both routes end in the same inventory phones, so the model sees one token
language. Known approximations (the inventory has no such sounds):

  retroflex ʈ ɖ ɽ -> t d ɾ     aspiration  kʰ -> k h    x ɣ q -> h ɡ k
  nasal vowel ẽ   -> e + n     ʋ -> v       ɟ c -> dʒ tʃ

Stress is dropped on purpose: the dataset's stress marks come from a rule
based phonemizer, and Roman Urdu text carries no stress to reproduce.

The UrduTTS ``roman`` column is NOT used. It is a vowel-less consonant
skeleton ("JS KY MSAL BNDGAN"), which cannot be spoken.
"""

import hashlib
import json
import re
import unicodedata

IPA_PREFIX = "ipa:"
VERSION = "urdu-1"

# ---- IPA (UrduTTS phonemes column) -> inventory --------------------------------------------

# (base, long) -> inventory phones. Anything missing falls through unchanged and
# becomes <unk>, which the annotator reports and rejects (so nothing hides).
_VOWELS = {
    ("a", False): ("ɐ",), ("a", True): ("ɑː",), ("ɑ", False): ("ɑː",), ("ɑ", True): ("ɑː",),
    ("ə", False): ("ə",), ("ə", True): ("ə",), ("ʌ", False): ("ʌ",), ("ʌ", True): ("ʌ",),
    ("ɛ", False): ("ɛ",), ("ɛ", True): ("ɛ",), ("e", False): ("eɪ",), ("e", True): ("eɪ",),
    ("i", False): ("i",), ("i", True): ("iː",), ("ɪ", False): ("ɪ",), ("ɪ", True): ("iː",),
    ("o", False): ("oː",), ("o", True): ("oː",), ("ɔ", False): ("ɔ",), ("ɔ", True): ("ɔː",),
    ("u", False): ("ʊ",), ("u", True): ("uː",), ("ʊ", False): ("ʊ",), ("ʊ", True): ("uː",),
    ("ɐ", False): ("ɐ",), ("ɐ", True): ("ɑː",),
}
_CONSONANTS = {
    "p": ("p",), "b": ("b",), "t": ("t",), "d": ("d",), "k": ("k",), "ɡ": ("ɡ",), "g": ("ɡ",),
    "f": ("f",), "s": ("s",), "z": ("z",), "ʃ": ("ʃ",), "ʒ": ("ʒ",), "h": ("h",),
    "m": ("m",), "n": ("n",), "l": ("l",), "j": ("j",), "w": ("w",), "v": ("v",),
    "r": ("ɾ",), "ɾ": ("ɾ",), "ɹ": ("ɹ",), "ʔ": ("ʔ",), "ð": ("ð",), "θ": ("θ",), "ŋ": ("ŋ",),
    "ɟ": ("dʒ",), "c": ("tʃ",), "ʂ": ("ʃ",), "ʐ": ("ʒ",), "ʋ": ("v",),
    "x": ("h",), "ɣ": ("ɡ",), "q": ("k",),
    "ʈ": ("t",), "ɖ": ("d",), "ɽ": ("ɾ",), "ɳ": ("n",), "ɭ": ("l",), "ɲ": ("n",), "ɫ": ("l",),
}
_IGNORED = set("ˈˌ.|‿͜͡")     # stress, syllable dots, tie bars
_COMBINING_NASAL = "̃"


def _ipa_word(word):
    """One IPA word -> inventory phones (list of str)."""
    word = unicodedata.normalize("NFD", word)
    units = []                           # [base, long, aspirated, nasal]
    for char in word:
        if char in _IGNORED:
            continue
        if char == "ː" and units:
            units[-1][1] = True
        elif char == "ʰ" and units:
            units[-1][2] = True
        elif char == _COMBINING_NASAL and units:
            units[-1][3] = True
        elif unicodedata.combining(char):
            continue
        else:
            units.append([char, False, False, False])
    phones = []
    for base, long, aspirated, nasal in units:
        if (base, long) in _VOWELS:
            phones.extend(_VOWELS[(base, long)])
            if nasal:
                phones.append("n")
        else:
            phones.extend(_CONSONANTS.get(base, (base,)))
            if aspirated:
                phones.append("h")
    return phones


def ipa_to_words(text):
    return [w for w in (_ipa_word(w) for w in text.split()) if w]


# ---- Roman Urdu -> inventory (rule based) ------------------------------------------------

# Very frequent words whose spelling varies or breaks the rules.
_LEXICON = {
    "hai": "h ɛ", "hay": "h ɛ", "he": "h ɛ", "hain": "h ɛ n", "hein": "h ɛ n", "hen": "h ɛ n",
    "mein": "m eɪ n", "main": "m eɪ n", "men": "m eɪ n", "me": "m eɪ n",
    "nahi": "n ɐ h iː", "nahin": "n ɐ h iː n", "nahee": "n ɐ h iː", "nai": "n ɐ h iː",
    "aur": "ɔ ɾ", "or": "ɔ ɾ", "ka": "k ɑː", "ki": "k iː", "ke": "k eɪ", "ko": "k oː",
    "se": "s eɪ", "ne": "n eɪ", "par": "p ɐ ɾ", "yeh": "j eɪ", "ye": "j eɪ", "woh": "v oː", "wo": "v oː",
    "to": "t oː", "bhi": "b iː", "hi": "h iː", "is": "ɪ s", "us": "ʊ s", "in": "ɪ n", "un": "ʊ n",
    "kya": "k j ɑː", "kyun": "k j ʊ n", "kyon": "k j ʊ n", "kuch": "k ʊ tʃ", "kuchh": "k ʊ tʃ",
    "tha": "t ɑː", "thi": "t iː", "the": "t eɪ", "thay": "t eɪ", "ho": "h oː", "hoon": "h uː n", "hun": "h uː n",
    "allah": "ɐ l l ɑː h", "mera": "m eɪ ɾ ɑː", "meri": "m eɪ ɾ iː", "mere": "m eɪ ɾ eɪ",
}

_DIGRAPHS = (
    ("chh", ("tʃ", "h")), ("kh", ("k", "h")), ("gh", ("ɡ", "h")), ("ch", ("tʃ",)), ("sh", ("ʃ",)),
    ("th", ("t", "h")), ("dh", ("d", "h")), ("ph", ("p", "h")), ("bh", ("b", "h")),
    ("jh", ("dʒ", "h")), ("zh", ("ʒ",)), ("ck", ("k",)),
    ("aa", ("ɑː",)), ("ee", ("iː",)), ("ii", ("iː",)), ("oo", ("uː",)), ("ai", ("ɛ",)), ("ay", ("eɪ",)),
    ("ey", ("eɪ",)), ("ei", ("ɛ",)), ("au", ("ɔ",)), ("ou", ("ɔ",)), ("ao", ("ɑː", "oː")),
)
_SINGLE = {
    "a": ("ɐ",), "e": ("eɪ",), "i": ("ɪ",), "o": ("oː",), "u": ("ʊ",),
    "b": ("b",), "p": ("p",), "t": ("t",), "d": ("d",), "k": ("k",), "g": ("ɡ",), "f": ("f",), "s": ("s",),
    "z": ("z",), "m": ("m",), "n": ("n",), "l": ("l",), "r": ("ɾ",), "h": ("h",), "j": ("dʒ",), "y": ("j",),
    "v": ("v",), "w": ("v",), "c": ("k",), "q": ("k",), "x": ("k", "s"),
}
_VOWEL_PHONES = {p for pair in _VOWELS.values() for p in pair}


def roman_to_phones(word):
    """One Roman Urdu word (letters only) -> inventory phones."""
    word = word.lower().replace("'", "")
    if word in _LEXICON:
        return _LEXICON[word].split()
    phones, i = [], 0
    while i < len(word):
        for letters, out in _DIGRAPHS:
            if word.startswith(letters, i):
                phones.extend(out)
                i += len(letters)
                break
        else:
            char, final = word[i], i == len(word) - 1
            if char == "a" and final and len(word) > 1:
                out = ("ɑː",)                      # kya, hamara, aaya: final a is long
            elif char == "i" and final and len(word) > 1:
                out = ("iː",)
            else:
                out = _SINGLE.get(char)
            if out:
                # geminate consonants ("achha") are written double; say them once
                if not (phones and out[0] == phones[-1] and out[0] not in _VOWEL_PHONES):
                    phones.extend(out)
            i += 1
    return phones


_TOKEN = re.compile(r"[A-Za-z']+|[,.?!;:\"()—…]")


def roman_pieces(text):
    from tiny_voca import text as base
    text = base.normalize_text(text).replace("-", " ")
    pieces, words = [], []
    for token in _TOKEN.findall(text):
        if token[0].isalpha() or token[0] == "'":
            phones = roman_to_phones(token)
            if phones:
                words.append(phones)
        else:
            if words:
                pieces.append((False, words))
                words = []
            pieces.append((True, token))
    if words:
        pieces.append((False, words))
    return pieces


# ---- tiny-voca integration -----------------------------------------------------------------

def _as_pieces(words):
    return [(False, [[(p, 0) for p in w] for w in words])]


def pieces(text):
    if text.startswith(IPA_PREFIX):
        return _as_pieces(ipa_to_words(text[len(IPA_PREFIX):]))
    return [(flag, value if flag else [[(p, 0) for p in w] for w in value]) for flag, value in roman_pieces(text)]


def _rules_digest():
    blob = json.dumps([_VOWELS_KEYS(), _CONSONANTS, _LEXICON, _DIGRAPHS, _SINGLE, VERSION],
                      sort_keys=True, ensure_ascii=False, default=list)
    return hashlib.sha256(blob.encode()).hexdigest()[:16]


def _VOWELS_KEYS():
    return {f"{k[0]}{int(k[1])}": v for k, v in _VOWELS.items()}


def install(text_module):
    """Replace tiny_voca.text's English front end with this one. Call BEFORE
    importing tiny_voca.annotate / train / infer, which bind ``phonemize`` by name."""
    def phonemize(text):
        if not isinstance(text, str) or not text.strip():
            raise ValueError("text must not be empty")
        return text_module.tokenize(pieces(text))

    def frontend_identity():
        return {"frontend_version": VERSION, "phonemizer": "tiny-urdu", "rules": _rules_digest(),
                "symbols": list(text_module.SYMBOLS)}

    def frontend_runtime():
        return {"phonemizer": "tiny-urdu-" + _rules_digest()}

    text_module.phonemize = phonemize
    text_module.frontend_identity = frontend_identity
    text_module.frontend_runtime = frontend_runtime

In [ ]:
%%writefile urdu/run.py
"""Run a tiny_voca module with the Urdu front end installed.

    python urdu/run.py tiny_voca.annotate --features ...
    python urdu/run.py tiny_voca.train ...
    python urdu/run.py tiny_voca.infer --checkpoint ... --text "aap kaise hain?"

The patch has to happen before the target module is imported, because
tiny-voca modules do ``from .text import phonemize``.
"""

import os
import runpy
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import frontend  # noqa: E402
from tiny_voca import text  # noqa: E402

frontend.install(text)

if __name__ == "__main__":
    if len(sys.argv) < 2:
        raise SystemExit(__doc__)
    module = sys.argv[1]
    sys.argv = [module] + sys.argv[2:]
    runpy.run_module(module, run_name="__main__", alter_sys=True)

In [ ]:
%%writefile urdu/prepare_data.py
"""Stream N UrduTTS clips, resample 44.1k -> 24k, write wavs + tiny-voca JSONL.

    python urdu/prepare_data.py --train 1000 --validation 32 --output data

Training text is "ipa:" + the dataset's `phonemes` column (see urdu/frontend.py
for why the vowel-less `roman` column is not used).
"""

import argparse
import json
from pathlib import Path

import numpy as np
import soundfile as sf
from scipy.signal import resample_poly

SR = 24000
SOURCE_SR = 44100
REPO = "kaab4321/UrduTTS"


def stream(split, count, min_s, max_s, out_dir, speaker, prefix):
    from datasets import Audio, load_dataset
    ds = load_dataset(REPO, split=split, streaming=True).cast_column("audio", Audio(decode=False))
    wav_dir = out_dir / "wavs"
    wav_dir.mkdir(parents=True, exist_ok=True)
    rows, skipped = [], 0
    for index, row in enumerate(ds):
        if len(rows) >= count:
            break
        phonemes = (row.get("phonemes") or "").strip()
        if not phonemes:
            skipped += 1
            continue
        wave, sr = sf.read(__import__("io").BytesIO(row["audio"]["bytes"]), dtype="float32")
        if wave.ndim > 1:
            wave = wave.mean(axis=1)
        seconds = len(wave) / sr
        if not min_s <= seconds <= max_s:
            skipped += 1
            continue
        if sr != SR:
            g = np.gcd(sr, SR)
            wave = resample_poly(wave, SR // g, sr // g).astype(np.float32)
        peak = float(np.max(np.abs(wave)))
        if peak > 0.99:
            wave = wave * (0.99 / peak)
        name = f"{prefix}{len(rows):05d}"
        sf.write(wav_dir / f"{name}.wav", wave, SR, subtype="PCM_16")
        rows.append({"id": name, "text": "ipa:" + phonemes, "speaker": speaker, "audio": f"wavs/{name}.wav",
                     "origin": "kaab4321/UrduTTS", "source_id": f"{split}:{index}",
                     "urdu": row.get("transcription", "").strip(), "seconds": round(seconds, 2)})
        if len(rows) % 100 == 0:
            print(f"[{split}] {len(rows)}/{count} (skipped {skipped})", flush=True)
    return rows


def median_f0(out_dir, rows, n=20):
    import librosa
    f0s = []
    for row in rows[:n]:
        wave, _ = sf.read(out_dir / row["audio"], dtype="float32")
        f0 = librosa.yin(wave, fmin=60, fmax=500, sr=SR, frame_length=2048)
        rms = librosa.feature.rms(y=wave, frame_length=2048, hop_length=512)[0][:len(f0)]
        f0s.extend(f0[:len(rms)][rms > np.percentile(rms, 60)])
    return float(np.median(f0s))


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("--train", type=int, default=1000)
    p.add_argument("--validation", type=int, default=32)
    p.add_argument("--output", type=Path, default=Path("data"))
    p.add_argument("--min-seconds", type=float, default=1.5)
    p.add_argument("--max-seconds", type=float, default=12.0, help="shorter clips = cheaper CPU training")
    p.add_argument("--speaker", choices=("auto", "F2", "M2"), default="auto",
                   help="which tiny-voca voice slot the Urdu speaker is written into (auto = by median pitch)")
    a = p.parse_args()
    a.output.mkdir(parents=True, exist_ok=True)

    train = stream("train", a.train, a.min_seconds, a.max_seconds, a.output, "F2", "tr")
    speaker = a.speaker
    if speaker == "auto":
        f0 = median_f0(a.output, train)
        speaker = "F2" if f0 > 165 else "M2"
        print(f"median f0 {f0:.0f} Hz -> voice slot {speaker}")
    val = stream("test", a.validation, a.min_seconds, a.max_seconds, a.output, speaker, "va")
    for row in train:
        row["speaker"] = speaker
    for name, rows in (("train", train), ("validation", val)):
        with (a.output / f"{name}.jsonl").open("w", encoding="utf-8") as f:
            for row in rows:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
    hours = sum(r["seconds"] for r in train) / 3600
    print(json.dumps({"train": len(train), "validation": len(val), "train_hours": round(hours, 2), "speaker": speaker}))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile urdu/make_init_checkpoint.py
"""Wrap the released inference weights as a starting point for `tiny_voca.train --init-from`.

The release (tiny-voca.pt) has the speech model only: no critics, optimizer or
mel head. train.py's --init-from wants a full training state, so this builds a
minimal one at step 0. The missing parts (mel head, critics) start fresh, which
is correct for a step-0 branch: critics are only used after --gan-start.

    python urdu/make_init_checkpoint.py --weights weights/tiny-voca.pt --output runs/init.pt
"""

import argparse
import json
from pathlib import Path

import torch

from tiny_voca.config import CHECKPOINT_FORMAT, Config


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("--weights", type=Path, required=True)
    p.add_argument("--output", type=Path, required=True)
    a = p.parse_args()
    released = torch.load(a.weights, map_location="cpu", weights_only=True)
    if released.get("format_version") != CHECKPOINT_FORMAT:
        raise SystemExit(f"unexpected checkpoint format {released.get('format_version')}")
    if released["config"] != Config().to_dict():
        raise SystemExit("released config differs from tiny_voca.config.Config() defaults")
    state = {
        "format_version": CHECKPOINT_FORMAT, "config": released["config"],
        "frontend": released["frontend"], "annotation": released.get("annotation"),
        "step": 0, "datasets": [], "recipe": {"gan_start": 0, "note": "English tiny-voca, inference weights only"},
        "system": {"model." + k: v for k, v in released["model"].items()},
    }
    a.output.parent.mkdir(parents=True, exist_ok=True)
    torch.save(state, a.output)
    print(json.dumps({"tensors": len(released["model"]), "output": str(a.output)}))


if __name__ == "__main__":
    main()

## 1. Data: stream clips, resample to 24 kHz

In [ ]:
!python urdu/prepare_data.py --train {N_TRAIN} --validation {N_VAL} --output data

## 2. Acoustic features

In [ ]:
!python urdu/run.py tiny_voca.prepare --inputs data/train.jsonl --split train --output features --speaker-column speaker
!python urdu/run.py tiny_voca.prepare --inputs data/validation.jsonl --split validation --output features --speaker-column speaker

## 3. Forced alignment (phoneme durations)

In [ ]:
!python urdu/run.py tiny_voca.annotate --features features --output annotations --splits train validation --dataset-revision UrduTTS-streamed --device cuda

## 4. Fine-tune from English tiny-voca

In [ ]:
!python urdu/make_init_checkpoint.py --weights weights/tiny-voca.pt --output runs/init.pt
!python urdu/run.py tiny_voca.train \
  --train annotations/train.jsonl --validation annotations/validation.jsonl \
  --output runs/urdu1 --init-from runs/init.pt --new-data --device cuda \
  --steps {STEPS} --max-seconds 86400 --learning-rate 2e-4 --warmup 200 \
  --gan-start {STEPS//2} --gan-ramp {STEPS//8} --teacher-hold {STEPS//6} --teacher-ramp {STEPS//3} --teacher-final 0 \
  --save-every 500 --keep-every 2000 --log-every 25 --validation-size {N_VAL}

## 5. Speak Roman Urdu

In [ ]:
import json, subprocess, IPython.display as ipd
voice = json.loads(open("data/train.jsonl").readline())["speaker"]
for i, t in enumerate(["aap kaise hain?", "mujhe chai pasand hai.", "shukriya, bohat shukriya!"]):
    subprocess.run(["python","urdu/run.py","tiny_voca.infer","--checkpoint","runs/urdu1/inference.pt",
                    "--voice",voice,"--text",t,"--output",f"out{i}.wav","--device","cuda"],check=True)
    print(t); ipd.display(ipd.Audio(f"out{i}.wav"))